## Statistical Testing
Author: Angela Palacino

The purpose of this notebook is to perform statistical testing to verify if the number of cases is equally distributed among age groups and localities.
The statistical test permoformed is Chi-squared, the hypothesis test of independence of the observed frequencies in the contingency table. 
- The null hypothesis: Drug abuse cases are independent of age group and locality — no association.
- The alternative hypothesis: Drug abuse cases are dependent on age group and/or locality — there is an association.

In [32]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

In [ ]:
# Read the parquet file with the processed data
master_df = pd.read_parquet(
    "/home/ange/DS_projects/psycho_substance_abuse/data/processed/master_population_cases.parquet.gzip",
    engine="pyarrow",
)

In [50]:
master_df

,Localidad,age_group,CASOS,Población
0,Antonio Nariño,Adolescence,94.0,4620
1,Antonio Nariño,Adulthood,597.0,39247
2,Antonio Nariño,Elderly,85.0,14532
3,Antonio Nariño,First infancy,0.0,9181
4,Antonio Nariño,Infancy,0.0,4843
...,...,...,...,...
115,Usme,Adulthood,1101.0,183713
116,Usme,Elderly,73.0,51206
117,Usme,First infancy,0.0,44135
118,Usme,Infancy,17.0,35975


In [ ]:
# Create the cases per 1000 habitants to have values according to the test (>5)
master_df["Cases per 1000"] = master_df["CASOS"] / master_df["Población"] * 1000

master_df.head()

,Localidad,age_group,CASOS,Población,Cases per 1000
0,Antonio Nariño,Adolescence,94.0,4620,20.346320
1,Antonio Nariño,Adulthood,597.0,39247,15.211354
2,Antonio Nariño,Elderly,85.0,14532,5.849160
3,Antonio Nariño,First infancy,0.0,9181,0.000000
4,Antonio Nariño,Infancy,0.0,4843,0.000000


To perfrom the statistical test is necessary to create a contingency table and remove any localities or age groups that have only 0 in their values.

In [ ]:
# Create contingency table, sort values by age group, drop first infancy age group
chi_squared_df = master_df.pivot(
    index="age_group", columns="Localidad", values="Cases per 1000"
)
custom_dict = {
    "First Infancy": 0,
    "Infancy": 1,
    "Adolescence": 2,
    "Youth": 3,
    "Adulthood": 4,
    "Elderly": 5,
}
chi_squared_df.sort_values(
    by="age_group", key=lambda x: x.map(custom_dict), ascending=False, inplace=True
)
chi_squared_df = chi_squared_df.drop("First infancy")
chi_squared_df

Localidad,Antonio Nariño,Barrios Unidos,Bosa,Candelaria,Chapinero,Ciudad Bolívar,Engativá,Fontibón,Kennedy,Mártires,Puente Aranda,Rafael Uribe,San Cristóbal,Santa Fe,Suba,Sumapaz,Teusaquillo,Tunjuelito,Usaquén,Usme
age_group,,,,,,,,,,,,,,,,,,,,
Elderly,5.849160,1.640387,1.281872,26.548673,2.391611,1.434935,1.208516,1.526407,1.542498,39.856027,4.638219,1.779449,1.676502,18.637399,0.684258,0.000000,1.783198,3.353789,1.094262,1.425614
Adulthood,15.211354,9.387308,6.464086,85.321848,9.110261,4.404906,4.914178,6.957493,5.324680,148.789091,19.847699,5.628692,8.048925,70.124022,3.179023,0.000000,10.161824,10.635572,3.622016,5.993043
Youth,36.795158,32.052511,32.242881,194.789955,55.586334,26.127881,21.759643,33.338848,20.352241,162.632375,60.096154,27.188428,30.274245,86.867538,18.949495,3.355705,64.655172,50.992410,19.687712,31.788327
Adolescence,20.346320,26.534480,45.356824,64.739884,27.999398,48.419236,17.760584,22.810956,18.011777,45.315081,26.569690,33.212706,60.390305,47.185709,33.371798,9.036145,61.577181,80.330124,16.363327,72.303360
Infancy,0.000000,0.000000,0.419490,1.230012,0.000000,0.235373,0.057053,0.115772,0.140915,0.182349,0.066747,0.338857,0.568864,0.650957,0.109630,0.000000,0.280662,0.152858,0.164971,0.472550


In [ ]:
# Perform the Chi squared test
chi2, p, dof, expected = chi2_contingency(chi_squared_df)
print(f"P-value: {p}")

P-value: 5.868976985009086e-63


The obtained p-value is lower than 0.05 which means it is safe to refect the null hypothesis that drug abuse cases are independent of age group and locality. Furthermore,the p-value is quite small to say there is a significant association between age group, locality, and drug abuse cases.